# FLARE v2 — Iterative Robust DRW Baseline + EVT on Real Data

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, pickle
import celerite
from celerite import terms
from scipy.optimize import minimize
from scipy.stats import genpareto
import os, glob, json
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],
    'font.size':12,'axes.labelsize':14,'xtick.labelsize':12,'ytick.labelsize':12,
    'legend.fontsize':11,'figure.dpi':150,'savefig.dpi':300,'savefig.bbox':'tight',
    'axes.linewidth':1.0,'xtick.major.width':1.0,'ytick.major.width':1.0,
    'xtick.major.size':5,'ytick.major.size':5,'xtick.minor.size':3,'ytick.minor.size':3,
    'xtick.direction':'in','ytick.direction':'in','xtick.top':True,'ytick.right':True,
    'axes.grid':False,'mathtext.fontset':'dejavuserif'})

C_DATA='#2c3e50'; C_MASKED='#e74c3c'; C_FIT='#2980b9'; C_ENV='#3498db'
C_THRESH='#e74c3c'; C_HIST='#1a8a7d'; C_ACC='#e67e22'; C_PUR='#8e44ad'

In [ ]:
# ── Paths ──
DATA_DIR  = r'path'
DRW_PKL   = r'path'
OUT_DIR   = r'path'
FIG_DIR   = r'path'
CAND_DIR  = r'path'
for d in [OUT_DIR, FIG_DIR, CAND_DIR]: os.makedirs(d, exist_ok=True)

lc_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
print(f'{len(lc_files)} light curves')

# ── MacLeod DRW params (dbID, tau, sigma) ──
with open(DRW_PKL, 'rb') as f: ml_drw = pickle.load(f)
print(f'MacLeod DRW params: {len(ml_drw)} quasars')
ml_drw = ml_drw.rename(columns={'tau':'tau_M10','sigma':'sigma_M10'})
ml_drw.head()

---
## Core Functions

In [ ]:
def fit_drw(t, y, yerr):
    kernel = terms.RealTerm(log_a=np.log(np.var(y)),
                           log_c=np.log(1/((t[-1]-t[0])/5)))
    kernel += terms.JitterTerm(log_sigma=np.log(np.median(yerr)))
    gp = celerite.GP(kernel, mean=np.mean(y), fit_mean=True)
    gp.compute(t, yerr)
    nll  = lambda p,y,gp: (gp.set_parameter_vector(p), -gp.log_likelihood(y))[1]
    gnll = lambda p,y,gp: (gp.set_parameter_vector(p), -gp.grad_log_likelihood(y)[1])[1]
    r = minimize(nll, gp.get_parameter_vector(), jac=gnll, args=(y,gp),
                 method='L-BFGS-B', bounds=gp.get_parameter_bounds())
    gp.set_parameter_vector(r.x)
    sig = np.sqrt(np.exp(gp.get_parameter('kernel:terms[0]:log_a')))
    tau = 1/np.exp(gp.get_parameter('kernel:terms[0]:log_c'))
    return gp, tau, sig, gp.log_likelihood(y)

def ou_resid(t, y, yerr, tau, sig, mu):
    N=len(t); dt=np.diff(t)
    z=np.zeros(N); mo=np.zeros(N); so=np.zeros(N); mo[0]=y[0]
    for i in range(1,N):
        d=np.exp(-dt[i-1]/tau)
        mo[i]=mu+(y[i-1]-mu)*d
        v=(sig**2*tau/2)*(1-np.exp(-2*dt[i-1]/tau))
        so[i]=np.sqrt(v)
        z[i]=(y[i]-mo[i])/np.sqrt(v+yerr[i]**2)
    return z, mo, so

def iter_drw(t, y, yerr, z_thr=3.0, max_it=10):
    N=len(t); mask=np.zeros(N,dtype=bool)
    h_tau,h_sig,h_nm=[],[],[]
    for it in range(max_it):
        ix=~mask
        if ix.sum()<15: break
        mu=np.mean(y[ix])
        try: gp,tau,sig,ll=fit_drw(t[ix],y[ix],yerr[ix])
        except: break
        h_tau.append(tau); h_sig.append(sig)
        z,mo,so=ou_resid(t,y,yerr,tau,sig,mu)
        nm=np.abs(z)>z_thr; nm[0]=nm[-1]=False; h_nm.append(int(nm.sum()))
        if np.array_equal(mask,nm): break
        mask=nm
    return dict(tau=tau,sigma_hat=sig,mean_mag=mu,gp=gp,mask=mask,z=z,
                max_z=np.max(np.abs(z)),n_iter=it+1,h_tau=h_tau,h_sig=h_sig,h_nm=h_nm)

---
## Batch Processing at Multiple Masking Thresholds

In [ ]:
def batch(files, z_thr=3.0):
    rows,mzl,fails=[],[],[]
    for f in tqdm(files, desc=f'z={z_thr}'):
        nm=os.path.basename(f); dbid=int(os.path.splitext(nm)[0])
        try:
            df=pd.read_csv(f)
            t,y,e=df['MJD_r'].values,df['mag_r'].values,df['magr_err'].values
            if len(t)<15: fails.append(nm); continue
            r=iter_drw(t,y,e,z_thr=z_thr)
            rows.append(dict(dbID=dbid,tau=r['tau'],sigma_hat=r['sigma_hat'],
                mean_mag=r['mean_mag'],max_z=r['max_z'],
                n_masked=int(r['mask'].sum()),n_epochs=len(t),
                n_iter=r['n_iter'],baseline=t[-1]-t[0]))
            mzl.append(r['max_z'])
        except: fails.append(nm)
    print(f'  {len(rows)} ok, {len(fails)} failed')
    return pd.DataFrame(rows), np.array(mzl), fails

ZT = [2.5, 3.0, 3.5, 4.0]
runs = {}
for zt in ZT:
    c,m,f = batch(lc_files, z_thr=zt)
    c.to_csv(os.path.join(OUT_DIR, f'catalog_z{zt}.csv'), index=False)
    runs[zt] = dict(cat=c, maxz=m, fails=f)

---
## EVT on Real Data

In [ ]:
def evt(maxz, pct=95, fap=0.01):
    N=len(maxz); u=np.percentile(maxz,pct)
    exc=maxz[maxz>u]-u; nu=len(exc)
    xi,_,beta=genpareto.fit(exc,floc=0)
    bnd=u+(beta/xi)*((N*fap/nu)**(-xi)-1) if xi!=0 else u-beta*np.log(N*fap/nu)
    return bnd, dict(xi=xi,beta=beta,u=u,nu=nu,N=N)

for zt in ZT:
    b,p=evt(runs[zt]['maxz'])
    nc=int((runs[zt]['maxz']>b).sum())
    runs[zt].update(bnd=b, prm=p, nc=nc)
    print(f'z_thr={zt}: boundary={b:.2f}σ, candidates={nc}, ξ={p["xi"]:.3f}')

---
## Figure 1 — Example Iterative DRW Fit

In [ ]:
c30=runs[3.0]['cat']; b30=runs[3.0]['bnd']
cids=c30[c30['max_z']>b30].sort_values('max_z',ascending=False)['dbID']
eid=cids.iloc[0] if len(cids)>0 else c30.iloc[0]['dbID']
df_ex=pd.read_csv(os.path.join(DATA_DIR,f'{int(eid)}.csv'))
t,y,e=df_ex['MJD_r'].values,df_ex['mag_r'].values,df_ex['magr_err'].values
res=iter_drw(t,y,e,z_thr=3.0)
m=res['mask']; gp=res['gp']; gp.compute(t[~m],e[~m])
tp=np.linspace(t.min(),t.max(),500)
mp,vp=gp.predict(y[~m],tp,return_var=True)

fig,(a1,a2)=plt.subplots(2,1,figsize=(7.5,5.5),
    gridspec_kw={'height_ratios':[3,1.4],'hspace':0.06},sharex=True)
a1.fill_between(tp,mp-2*np.sqrt(vp),mp+2*np.sqrt(vp),color=C_ENV,alpha=0.12,label=r'$2\sigma$ envelope')
a1.plot(tp,mp,color=C_FIT,lw=1.2,alpha=0.8,label='DRW fit')
a1.errorbar(t[~m],y[~m],yerr=e[~m],fmt='o',color=C_DATA,ms=3.5,elinewidth=0.8,capsize=0,zorder=5,label='Data')
if m.any(): a1.errorbar(t[m],y[m],yerr=e[m],fmt='D',color=C_MASKED,ms=5,elinewidth=0.8,capsize=0,zorder=6,label='Masked')
a1.invert_yaxis(); a1.set_ylabel(r'$r$ (mag)')
a1.legend(loc='upper right',framealpha=0.9,edgecolor='none',fontsize=10)
a1.text(0.02,0.92,rf'Quasar {int(eid)}   $\tau={res["tau"]:.0f}$ d   $\hat{{\sigma}}={res["sigma_hat"]:.3f}$',
    transform=a1.transAxes,fontsize=10,va='top',bbox=dict(boxstyle='round,pad=0.3',fc='white',ec='#ccc',alpha=0.9))
zv=res['z']
for i in range(len(t)):
    a2.plot(t[i],zv[i],'D' if m[i] else 'o',color=C_MASKED if m[i] else C_DATA,ms=5 if m[i] else 3.5,zorder=5)
a2.axhline(3,color=C_ACC,ls='--',lw=0.9,alpha=0.7); a2.axhline(-3,color=C_ACC,ls='--',lw=0.9,alpha=0.7)
a2.axhline(0,color='#999',ls='-',lw=0.5)
a2.set_xlabel('Relative MJD (days)'); a2.set_ylabel(r'$z_i$'); a2.set_xlim(t.min()-50,t.max()+50)
plt.savefig(os.path.join(FIG_DIR,'fig_example_iterative_drw.pdf')); plt.show()

---
## Figure 2 — EVT: GPD Fit & Detection Boundary

In [ ]:
mz=runs[3.0]['maxz']; prm=runs[3.0]['prm']
u,xi,beta=prm['u'],prm['xi'],prm['beta']; exc=mz[mz>u]-u

fig,(a1,a2)=plt.subplots(1,2,figsize=(7.5,3.5))
a1.hist(exc,bins=30,density=True,color=C_HIST,ec='white',lw=0.3,alpha=0.85,label='Empirical excesses')
xg=np.linspace(0,exc.max()*1.1,200)
a1.plot(xg,genpareto.pdf(xg,xi,scale=beta),color=C_THRESH,lw=2,label='GPD fit')
a1.set_xlabel(r'Excess over threshold ($u$)'); a1.set_ylabel('Probability density')
a1.legend(framealpha=0.9,edgecolor='none')
a1.text(0.05,0.92,'(a)',transform=a1.transAxes,fontsize=13,weight='bold')

a2.hist(mz,bins=50,color=C_HIST,ec='white',lw=0.3,alpha=0.85)
a2.axvline(b30,color=C_THRESH,ls='--',lw=1.8,label=rf'Threshold ({b30:.2f}$\sigma$)')
a2.set_yscale('log'); a2.set_xlabel(r'Calibrated max $|z|$'); a2.set_ylabel('Frequency')
a2.legend(framealpha=0.9,edgecolor='none')
a2.text(0.05,0.92,'(b)',transform=a2.transAxes,fontsize=13,weight='bold')
a2.text(0.95,0.75,f'{runs[3.0]["nc"]} candidates',transform=a2.transAxes,ha='right',fontsize=11,
    bbox=dict(boxstyle='round,pad=0.3',fc='#fff3cd',ec='#d4a017',alpha=0.95))
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'fig_evt_real_data.pdf')); plt.show()

---
## Figure 3 — EVT Stability Across Percentiles

In [ ]:
pcts=np.arange(90,99); sx,sb,sn=[],[],[]
for p in pcts:
    b,pr=evt(runs[3.0]['maxz'],pct=p)
    sx.append(pr['xi']); sb.append(b); sn.append(pr['nu'])

fig,axes=plt.subplots(1,3,figsize=(7.5,2.8))
for ax,yv,yl,lb in zip(axes,[sx,sb,sn],[r'$\xi$',r'Boundary ($\sigma$)','Number of excesses'],'abc'):
    ax.plot(pcts,yv,'o-',color=C_FIT,ms=5,lw=1.2)
    ax.set_xlabel('Threshold percentile'); ax.set_ylabel(yl)
    ax.text(0.05,0.92,f'({lb})',transform=ax.transAxes,fontsize=13,weight='bold')
axes[0].axhline(np.median(sx),color=C_THRESH,ls='--',lw=0.8,alpha=0.5)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'fig_evt_stability.pdf')); plt.show()

---
## Figure 4 — Robustness to Masking Threshold

In [ ]:
za=np.array(ZT)
ea=[runs[z]['bnd'] for z in ZT]; ca=[runs[z]['nc'] for z in ZT]
ta=[runs[z]['cat']['tau'].median() for z in ZT]
sa=[runs[z]['cat']['sigma_hat'].median() for z in ZT]

fig,axes=plt.subplots(1,4,figsize=(7.5,2.8))
for ax,(yv,yl,mk,cl),lb in zip(axes,
    [(ea,r'EVT boundary ($\sigma$)','o-',C_FIT),
     (ca,'Candidates','s-',C_ACC),
     (ta,r'Median $\tau$ (d)','o-',C_FIT),
     (sa,r'Median $\hat{\sigma}$','o-',C_PUR)],'abcd'):
    ax.plot(za,yv,mk,color=cl,ms=6,lw=1.3)
    ax.set_xlabel(r'$z_{\rm mask}$'); ax.set_ylabel(yl)
    ax.text(0.08,0.92,f'({lb})',transform=ax.transAxes,fontsize=13,weight='bold')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'fig_robustness_zmask.pdf')); plt.show()

# Summary table
print(f"{'z_mask':>6} | {'EVT σ':>8} | {'Cand':>5} | {'med τ':>7} | {'med σ̂':>8} | {'med mask':>8} | {'med iter':>8}")
print('-'*70)
for zt in ZT:
    c=runs[zt]['cat']
    print(f"{zt:>6.1f} | {runs[zt]['bnd']:>8.2f} | {runs[zt]['nc']:>5d} | "
          f"{c['tau'].median():>7.0f} | {c['sigma_hat'].median():>8.4f} | "
          f"{c['n_masked'].median():>8.0f} | {c['n_iter'].median():>8.0f}")

---
## Figure 5 — Candidate Overlap Heatmap

In [ ]:
cs={zt: set(runs[zt]['cat'][runs[zt]['cat']['max_z']>runs[zt]['bnd']]['dbID']) for zt in ZT}
common=set.intersection(*cs.values()); union=set.union(*cs.values())
print(f'Common across all: {len(common)},  Union: {len(union)}')

n=len(ZT); om=np.array([[len(cs[z1]&cs[z2]) for z2 in ZT] for z1 in ZT])
fig,ax=plt.subplots(figsize=(4,3.5))
im=ax.imshow(om,cmap='YlOrRd',aspect='auto')
for i in range(n):
    for j in range(n):
        ax.text(j,i,str(om[i,j]),ha='center',va='center',fontsize=12,
                color='white' if om[i,j]>om.max()*0.6 else 'black')
ax.set_xticks(range(n)); ax.set_xticklabels([str(z) for z in ZT])
ax.set_yticks(range(n)); ax.set_yticklabels([str(z) for z in ZT])
ax.set_xlabel(r'$z_{\rm mask}$'); ax.set_ylabel(r'$z_{\rm mask}$')
plt.colorbar(im,ax=ax,label='Shared candidates',shrink=0.8)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'fig_candidate_overlap.pdf')); plt.show()

robust=c30[c30['dbID'].isin(common)].sort_values('max_z',ascending=False)
robust.to_csv(os.path.join(OUT_DIR,'robust_candidates.csv'),index=False)

---
## Figure 6 — Convergence & Parameter Distributions

In [ ]:
fig,((a1,a2),(a3,a4))=plt.subplots(2,2,figsize=(7.5,6.5))

a1.hist(c30['n_iter'],bins=np.arange(0.5,c30['n_iter'].max()+1.5),color=C_HIST,ec='white',lw=0.5,alpha=0.85)
a1.set_xlabel('Iterations'); a1.set_ylabel('Quasars')
a1.text(0.05,0.92,'(a)',transform=a1.transAxes,fontsize=13,weight='bold')
a1.text(0.95,0.85,f'median={c30["n_iter"].median():.0f}',transform=a1.transAxes,ha='right',fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3',fc='white',ec='#ccc',alpha=0.9))

a2.hist(c30['n_masked'],bins=np.arange(-0.5,c30['n_masked'].max()+1.5),color=C_ACC,ec='white',lw=0.5,alpha=0.85)
a2.set_xlabel('Masked epochs'); a2.set_ylabel('Quasars')
a2.text(0.05,0.92,'(b)',transform=a2.transAxes,fontsize=13,weight='bold')
a2.text(0.95,0.85,f'median={c30["n_masked"].median():.0f}',transform=a2.transAxes,ha='right',fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3',fc='white',ec='#ccc',alpha=0.9))

a3.hist(np.log10(c30['tau']),bins=50,color=C_FIT,ec='white',lw=0.3,alpha=0.85)
a3.axvline(np.log10(c30['tau'].median()),color=C_THRESH,ls='--',lw=1.2,
    label=rf'median={c30["tau"].median():.0f} d')
a3.set_xlabel(r'$\log_{10}\,\tau$ (days)'); a3.set_ylabel('Quasars')
a3.legend(framealpha=0.9,edgecolor='none')
a3.text(0.05,0.92,'(c)',transform=a3.transAxes,fontsize=13,weight='bold')

a4.hist(np.log10(c30['sigma_hat']),bins=50,color=C_PUR,ec='white',lw=0.3,alpha=0.85)
a4.axvline(np.log10(c30['sigma_hat'].median()),color=C_THRESH,ls='--',lw=1.2,
    label=rf'median={c30["sigma_hat"].median():.4f}')
a4.set_xlabel(r'$\log_{10}\,\hat{\sigma}$'); a4.set_ylabel('Quasars')
a4.legend(framealpha=0.9,edgecolor='none')
a4.text(0.05,0.92,'(d)',transform=a4.transAxes,fontsize=13,weight='bold')

plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'fig_convergence_and_params.pdf')); plt.show()

---
## Figure 7 — Comparison with MacLeod et al. (2010) DRW Parameters

In [ ]:
comp = c30.merge(ml_drw, on='dbID', how='inner')
print(f'Matched for comparison: {len(comp)}/{len(c30)}')

# Filter out edge cases: MacLeod tau=1e-10 or sigma=1e-10 are failed fits
good = (comp['tau_M10'] > 1) & (comp['sigma_M10'] > 1e-5) & (comp['tau'] > 1) & (comp['sigma_hat'] > 1e-5)
cg = comp[good]
print(f'After filtering bad fits: {len(cg)}')

fig,(a1,a2)=plt.subplots(1,2,figsize=(7.5,3.5))

# (a) tau comparison
a1.scatter(np.log10(cg['tau_M10']),np.log10(cg['tau']),s=3,alpha=0.15,
           color=C_FIT,rasterized=True)
lims=[0.5,5]
a1.plot(lims,lims,'--',color=C_THRESH,lw=1,label='1:1')
a1.set_xlabel(r'$\log_{10}\,\tau_{\rm M10}$ (days)')
a1.set_ylabel(r'$\log_{10}\,\tau_{\rm this\ work}$ (days)')
a1.set_xlim(lims); a1.set_ylim(lims)
a1.legend(framealpha=0.9,edgecolor='none')
a1.text(0.05,0.92,'(a)',transform=a1.transAxes,fontsize=13,weight='bold')

# (b) sigma comparison
a2.scatter(np.log10(cg['sigma_M10']),np.log10(cg['sigma_hat']),s=3,alpha=0.15,
           color=C_PUR,rasterized=True)
lims2=[-2.5,0.5]
a2.plot(lims2,lims2,'--',color=C_THRESH,lw=1,label='1:1')
a2.set_xlabel(r'$\log_{10}\,\hat{\sigma}_{\rm M10}$')
a2.set_ylabel(r'$\log_{10}\,\hat{\sigma}_{\rm this\ work}$')
a2.set_xlim(lims2); a2.set_ylim(lims2)
a2.legend(framealpha=0.9,edgecolor='none')
a2.text(0.05,0.92,'(b)',transform=a2.transAxes,fontsize=13,weight='bold')

plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'fig_macleod_drw_comparison.pdf')); plt.show()

# Print correlation stats
from scipy.stats import pearsonr, spearmanr
r_tau = spearmanr(np.log10(cg['tau_M10']), np.log10(cg['tau']))
r_sig = spearmanr(np.log10(cg['sigma_M10']), np.log10(cg['sigma_hat']))
print(f'Spearman correlation (log τ):  ρ={r_tau.correlation:.3f},  p={r_tau.pvalue:.2e}')
print(f'Spearman correlation (log σ̂): ρ={r_sig.correlation:.3f},  p={r_sig.pvalue:.2e}')

---
## Figure 8 — Candidate Light Curves (r-band)

In [ ]:
# Get candidates from z_thr=3.0
cands = c30[c30['max_z']>b30].sort_values('max_z',ascending=False).reset_index(drop=True)
cands.to_csv(os.path.join(OUT_DIR,'flare_candidates.csv'),index=False)
print(f'Threshold: {b30:.2f}σ,  Candidates: {len(cands)}')

n_cands = len(cands)
ncols = 4
nrows = int(np.ceil(n_cands / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(7.5, 2.2*nrows))
axes = axes.flatten() if n_cands > ncols else [axes] if n_cands==1 else axes

for idx, (_, row) in enumerate(cands.iterrows()):
    ax = axes[idx]
    dbid = int(row['dbID'])
    fpath = os.path.join(DATA_DIR, f'{dbid}.csv')
    df = pd.read_csv(fpath)
    t,y,e = df['MJD_r'].values, df['mag_r'].values, df['magr_err'].values

    # Rerun to get mask
    res = iter_drw(t, y, e, z_thr=3.0)
    m = res['mask']

    ax.errorbar(t[~m],y[~m],yerr=e[~m],fmt='o',color=C_DATA,ms=2,elinewidth=0.5,capsize=0)
    if m.any():
        ax.errorbar(t[m],y[m],yerr=e[m],fmt='D',color=C_MASKED,ms=3,elinewidth=0.5,capsize=0)
    ax.invert_yaxis()
    ax.set_title(f'{dbid}', fontsize=9, pad=2)
    ax.tick_params(labelsize=7)
    if idx >= (nrows-1)*ncols: ax.set_xlabel('MJD',fontsize=8)
    if idx % ncols == 0: ax.set_ylabel(r'$r$',fontsize=8)

    # Copy light curve to candidate folder
    df.to_csv(os.path.join(CAND_DIR, f'{dbid}_r.csv'), index=False)

# Hide unused axes
for idx in range(n_cands, len(axes)): axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'fig_candidate_lightcurves_r.pdf'))
plt.show()
print(f'\nCandidate r-band light curves copied to {CAND_DIR}')

---
## Save Summary

In [ ]:
summary = {
    'robustness': {
        str(zt): dict(
            evt_bnd=float(runs[zt]['bnd']), n_cand=runs[zt]['nc'],
            xi=float(runs[zt]['prm']['xi']),
            med_tau=float(runs[zt]['cat']['tau'].median()),
            med_sigma=float(runs[zt]['cat']['sigma_hat'].median())
        ) for zt in ZT
    },
    'candidates_z3.0': {
        'threshold': float(b30),
        'n_candidates': int(len(cands)),
        'common_across_all_thresholds': int(len(common)),
        'dbIDs': [int(x) for x in cands['dbID'].tolist()]
    },
    'macleod_comparison': {
        'n_matched': int(len(cg)),
        'spearman_log_tau': float(r_tau.correlation),
        'spearman_log_sigma': float(r_sig.correlation)
    }
}
with open(os.path.join(OUT_DIR,'summary.json'),'w') as f:
    json.dump(summary,f,indent=2)

print(f'Figures → {FIG_DIR}')
print(f'Data    → {OUT_DIR}')
print(f'Candidate LCs → {CAND_DIR}')
print(f'\nDone.')